[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-11-transformers.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 11 — Encoder and decoder transformers

**MSIS · AME 5053 · Week 11 · 3 hours**

Session 31 made a claim that is easy to say and hard to believe: an encoder and a decoder are the
same block, and the only difference between them is whether attention is allowed to look ahead.
Everything else — the layer norms, the residual connections, the feedforward layer, the stacking —
is identical.

This lab takes that literally. You will write **one** block, from SLP3's equations, and then use it
twice: once as an encoder that classifies reviews, and once as a decoder that generates text. The
only edit between the two is a mask.

**By the end of this lab you will be able to:**

1. Write a transformer block from equations (8.21)–(8.31), in the pre-norm form SLP3 uses
2. Prove that a causal mask does what it claims, rather than trusting that it does
3. Build an encoder-only classifier and a decoder-only language model from the same block
4. Explain why a transformer trained from scratch here loses to methods from earlier in the course

---

## Part 0 — Setup

The classifier half uses the same corpus and the same 1,500/500 split as labs 6, 7, 8 and 9, so
the number it produces belongs in the same table as theirs.

In [ ]:
%pip install -q nltk scikit-learn

import nltk
ok = nltk.download("movie_reviews")
print("movie_reviews downloaded:", ok)

import math, re, time
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
print("torch", torch.__version__)
print("Done.")

### Where this will run

This is the lab a GPU helps most: attention is dense matrix multiplication, which is what GPUs are
built for. If Colab has given you one (Runtime → Change runtime type → T4 GPU) the models train
there. Everything still runs without one — the numbers below were measured on CPU.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("training on:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

> **Save your own copy now:** File → Save a copy in Drive.

In [ ]:
from nltk.corpus import movie_reviews

PAD, UNK = 0, 1
MAX_LEN = 300
token_re = re.compile(r"[a-z]+")

def tokenize(text):
    return token_re.findall(text.lower())

ids = movie_reviews.fileids()
texts = [movie_reviews.raw(i) for i in ids]
labels = np.array([1 if i.startswith("pos") else 0 for i in ids])
Xtr_text, Xte_text, ytr, yte = train_test_split(
    texts, labels, test_size=500, random_state=42, stratify=labels)

tok_train = [tokenize(t) for t in Xtr_text]
tok_test = [tokenize(t) for t in Xte_text]

counts = Counter(w for d in tok_train for w in d)
itos = ["<pad>", "<unk>"] + [w for w, n in counts.most_common() if n >= 2][:20000]
stoi = {w: i for i, w in enumerate(itos)}

def encode(docs, max_len=MAX_LEN):
    seqs = [[stoi.get(w, UNK) for w in d][-max_len:] or [UNK] for d in docs]
    lens = torch.tensor([len(s) for s in seqs])
    X = torch.zeros(len(seqs), int(lens.max()), dtype=torch.long)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)
    return X, lens

Xtr, ltr = encode(tok_train)
Xte, lte = encode(tok_test)
print("train", tuple(Xtr.shape), "· test", tuple(Xte.shape), "· vocabulary", len(itos))

# Verified output:
#   train (1500, 300) · test (500, 300) · vocabulary 20002

---

## Part 1 — One block

Session 31 gave six equations for a transformer block. In the pre-norm form SLP3 uses throughout
the chapter, they are:

    t1 = LayerNorm(x)
    t2 = MultiHeadAttention(t1)
    t3 = t2 + x
    t4 = LayerNorm(t3)
    t5 = FFN(t4)
    h  = t5 + t3

Two things are worth noticing before writing it. The **residual additions** — `+ x` and `+ t3` —
are what let a stack of these train at all; they are the third answer this course has seen to the
problem of a gradient having to survive many steps, after the LSTM's gates and lab 9's pooling.
And the normalisation sits *before* each sublayer, not after: SLP3 says "throughout the chapter,
we use the prenorm version", while Vaswani's paper puts it after. Session 31 named that
disagreement; we follow SLP3.

In [ ]:
class Block(nn.Module):
    """One transformer block: attention sublayer, then feedforward sublayer,
    each normalised on the way in and added back on the way out."""

    def __init__(self, d, heads, ff_mult=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, ff_mult * d), nn.GELU(),
            nn.Linear(ff_mult * d, d), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None, attn_mask=None):
        # YOUR CODE HERE — the six equations above, in order.
        # self.attn returns a (output, weights) pair; you want the output.
        #   t2, _ = self.attn(t1, t1, t1, key_padding_mask=..., attn_mask=..., need_weights=False)
        # Note all three of its first arguments are the same tensor. That is what makes
        # this SELF-attention: the queries, keys and values all come from one sequence.
        pass

### Position

Attention has no idea what order its inputs came in — session 30 established that, and it is why
positional information has to be added to the embeddings rather than implied by them. We use the
sinusoidal encoding.

In [ ]:
def positional_encoding(n, d):
    """Session 30's sinusoids: even dimensions get sin, odd get cos."""
    pos = torch.arange(n).unsqueeze(1).float()
    i = torch.arange(0, d, 2).float()
    angles = pos / torch.pow(10000, i / d)
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(angles)
    pe[:, 1::2] = torch.cos(angles)
    return pe

pe = positional_encoding(300, 128)
print("positional encoding:", tuple(pe.shape))
assert not torch.allclose(pe[0], pe[1]), "different positions must get different vectors"
print("Position 0 and position 1 differ, as they must.")

---

## Part 2 — The mask, tested rather than trusted

Session 31's claim is that the *only* difference between an encoder and a decoder is whether
attention may look ahead. A decoder must not: at generation time the words to the right do not
exist yet, so a model trained with access to them would be trained on information it will never
have.

The mask is an upper-triangular matrix of `True` above the diagonal, marking the positions each
query is forbidden to attend to.

In [ ]:
def causal_mask(n):
    """True where attention is FORBIDDEN, for a sequence of length n.

    Position i may attend to positions 0..i, and to nothing after i.
    """
    # YOUR CODE HERE
    # torch.triu(x, diagonal=1) keeps everything strictly ABOVE the main diagonal,
    # which is exactly the set of positions each query must not see.
    # Return a bool tensor of shape (n, n).
    pass

In [ ]:
print(causal_mask(5).int())
print("\nRow i shows what position i may NOT see: row 0 may see only itself,")
print("row 4 may see everything up to and including position 4.")

### The test

Rather than trust the mask, measure it. Take a model, run a sequence through it, and note the
prediction at position 10. Then change a token at position **40** — thirty steps in the future —
and look at position 10 again.

With a causal mask, position 10 cannot see position 40, so its prediction must be *identical*. Not
close: identical. Without the mask, it will move.

The model below is untrained, deliberately. This is a property of the architecture, not something
a model learns to respect.

In [ ]:
class DecoderLM(nn.Module):
    """A decoder-only language model: the same Block, with a causal mask."""

    def __init__(self, vocab, d=128, heads=4, layers=2, max_len=64, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.register_buffer("pe", positional_encoding(max_len, d))
        self.blocks = nn.ModuleList([Block(d, heads, dropout=dropout) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)
        self.d = d

    def forward(self, x, causal=True):
        T = x.size(1)
        h = self.emb(x) * math.sqrt(self.d) + self.pe[:T]
        # the mask must live on the same device as the activations it applies to
        mask = causal_mask(T).to(x.device) if causal else None
        for block in self.blocks:
            h = block(h, attn_mask=mask)
        return self.head(self.ln(h))

In [ ]:
torch.manual_seed(0)
probe = DecoderLM(1000, max_len=64).eval()
x = torch.randint(2, 1000, (1, 64))

for causal in (True, False):
    with torch.no_grad():
        before = probe(x, causal=causal)[0, 10].clone()
        x_edited = x.clone()
        x_edited[0, 40] = (x_edited[0, 40] + 7) % 1000     # change a token 30 steps later
        after = probe(x_edited, causal=causal)[0, 10]
        change = (before - after).abs().max().item()
    verdict = "unchanged" if change == 0 else "CHANGED — it saw the future"
    print(f"causal={causal!s:5}  max change at position 10: {change:.3e}   {verdict}")

# Verified output:
#   causal=True   max change at position 10: 0.000e+00   unchanged
#   causal=False  max change at position 10: 1.843e-03   CHANGED — it saw the future

In [ ]:
with torch.no_grad():
    before = probe(x, causal=True)[0, 10].clone()
    x_edited = x.clone()
    x_edited[0, 40] = (x_edited[0, 40] + 7) % 1000
    after = probe(x_edited, causal=True)[0, 10]
assert torch.equal(before, after), "a causal model must not see position 40 from position 10"
print("Exactly equal, to the last bit. The mask is not approximately working.")

That exact zero is the whole encoder/decoder distinction. An encoder omits the mask because
deciding what a word *means* is helped by the words on both sides of it; a decoder includes it
because generating the next word cannot depend on words that have not been generated.

**What this does not show.** You might expect that a model allowed to see the future would find
predicting the next word trivial, and score a perplexity near 1. It does not. Reading the answer
requires the attention to *learn* to look one position ahead, and that is a pattern like any other.
Trained for 10 epochs, the unmasked model reaches a perplexity of 128.62 against the causal
model's 156.30 — better, and the gap widens as training goes on, but nothing like a
collapse. The leak is real and it is gradual.

---

## Part 3 — The same block as an encoder

No mask, and one addition: the block stack produces a vector per token, and a classifier needs one
vector per document. We average over the real tokens, exactly as lab 9 did with its LSTM states.

In [ ]:
class EncoderClassifier(nn.Module):
    def __init__(self, vocab, d=128, heads=4, layers=4, max_len=MAX_LEN, n_classes=2, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=PAD)
        self.register_buffer("pe", positional_encoding(max_len, d))
        self.blocks = nn.ModuleList([Block(d, heads, dropout=dropout) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_classes)
        self.d = d

    def forward(self, x, lens=None):
        pad = (x == PAD)
        h = self.emb(x) * math.sqrt(self.d) + self.pe[:x.size(1)]
        for block in self.blocks:
            h = block(h, key_padding_mask=pad)      # no attn_mask: the encoder may look ahead
        h = self.ln(h)
        keep = (~pad).unsqueeze(-1).float()
        return self.head((h * keep).sum(1) / keep.sum(1))

model = EncoderClassifier(len(itos))
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"of which embeddings: {model.emb.weight.numel():,}")

Most of those parameters are the embedding table — one 128-dimensional vector for each of 20,002
words. The four blocks are a small fraction of the total, which is worth knowing before drawing
conclusions about "model size" from a parameter count.

In [ ]:
def evaluate(model, X, lens, y, bs=32):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(y), bs):
            preds.append(model(X[i:i + bs].to(DEVICE)).argmax(1).cpu())
    return float((torch.cat(preds).numpy() == y).mean())


def train_classifier(epochs=8, bs=32, lr=3e-4, seed=42):
    torch.manual_seed(seed)
    model = EncoderClassifier(len(itos)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()
    y = torch.tensor(ytr).to(DEVICE)
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(ytr))
        for i in range(0, len(ytr), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            loss = loss_fn(model(Xtr[idx].to(DEVICE)), y[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        acc = evaluate(model, Xte, lte, yte)
        print(f"  epoch {ep + 1}: test accuracy {acc:.4f}  ({time.time() - t0:.0f}s)")
    return model

classifier = train_classifier()

### The row it earns

| representation | classifier | accuracy |
|---|---|---|
| raw counts (lab 6) | Naive Bayes | 0.809 |
| sublinear tf-idf (lab 8) | Logistic Regression | **0.865** |
| GloVe averaged (lab 8) | Logistic Regression | 0.724 |
| LSTM, GloVe init (lab 9) | its own head | 0.773 |
| **transformer encoder, from scratch (this lab)** | its own head | **0.7620** |

The transformer comes **last among the neural models**, and well behind tf-idf.

This is worth sitting with rather than explaining away. The transformer is the architecture behind
every system in sessions 32 to 36; it is not a worse idea than an LSTM. What it is, here, is an
architecture with very little built into it. An LSTM has recurrence built in — it processes words
in order because it cannot do otherwise. A transformer has to *learn* that order matters, from
data, having been handed position vectors and told to work it out. On 1,500 documents there is not
enough data to learn it.

Session 32's answer, and lab 12's, is to do the learning somewhere else: train on billions of
words first, then bring the result to these 1,500 documents. Lab 12 measures exactly that, on this
same split, and the ordering in the table above changes.

---

## Part 4 — The same block as a decoder

Now the other use. A language model over the same corpus: predict each word from the words before
it, which is session 11's objective and session 14's perplexity, with a transformer in place of
counts.

In [ ]:
def language_model_data(max_len=64, vocab_cap=5000, seed=42):
    docs = [tokenize(movie_reviews.raw(i)) for i in movie_reviews.fileids()]
    counts = Counter(w for d in docs for w in d)
    lm_itos = ["<pad>", "<unk>"] + [w for w, n in counts.most_common() if n >= 5][:vocab_cap]
    lm_stoi = {w: i for i, w in enumerate(lm_itos)}
    stream = [lm_stoi.get(w, UNK) for d in docs for w in d]
    n = (len(stream) - 1) // max_len
    X = torch.tensor([stream[i * max_len:(i + 1) * max_len] for i in range(n)])
    Y = torch.tensor([stream[i * max_len + 1:(i + 1) * max_len + 1] for i in range(n)])
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X), generator=g)
    X, Y = X[perm], Y[perm]
    return X[:-200], Y[:-200], X[-200:], Y[-200:], lm_itos

lm_Xtr, lm_Ytr, lm_Xte, lm_Yte, lm_itos = language_model_data()
print(f"windows: {len(lm_Xtr)} train, {len(lm_Xte)} test · vocabulary {len(lm_itos)}")
print("Y is X shifted by one — the next-word objective, unchanged since session 11.")

In [ ]:
def train_lm(causal=True, epochs=10, bs=32, lr=3e-4, seed=42):
    torch.manual_seed(seed)
    model = DecoderLM(len(lm_itos), max_len=64).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(lm_Xtr))
        for i in range(0, len(lm_Xtr), bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            logits = model(lm_Xtr[idx].to(DEVICE), causal=causal)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)),
                           lm_Ytr[idx].reshape(-1).to(DEVICE))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            lg = model(lm_Xte.to(DEVICE), causal=causal)
            te = loss_fn(lg.reshape(-1, lg.size(-1)),
                         lm_Yte.reshape(-1).to(DEVICE)).item()
        print(f"  epoch {ep + 1}: test perplexity {math.exp(te):.2f}  ({time.time() - t0:.0f}s)")
    return model

lm = train_lm(causal=True, epochs=10)

Perplexity is session 14's measure, and the number is poor by the standards of a real language
model — this is 5,000 words of vocabulary, a two-block model, and a corpus of film reviews. What
matters is that the same block, with one mask added, is now doing a completely different job.

---

## What this lab established

1. A transformer block is six equations, and the residual additions are the reason a stack of them
   trains.
2. An encoder and a decoder differ by a mask, and the mask can be *proved* to work: editing the
   future changes an earlier prediction by exactly zero.
3. Seeing the future is a leak rather than a shortcut — the unmasked model does better and better
   as it learns to exploit it, but it does not get the answer for free.
4. A transformer trained from scratch on 1,500 documents loses to tf-idf, and to the LSTM. The
   architecture is not the problem; the amount of data is.

Lab 12 takes the last point seriously: instead of training a transformer, it takes one that has
already been trained on billions of words and adapts it to this task.